# Put-Call Parity
$$Call+Ke^{-rT}=Put+S_0$$
Call 與 Put 價格由市場決定，當上式不成立時，會出現套利機會。

# Black-Scholes Model
### Assumptions:
- European-style options.
- Percentage changes in the stock prices are normally distributed.
- The stock price at a future time $T$, $S_T$, is log-normally distributed(Geometric Brownian Motion).
- Continuously compounded return is used.
- No transaction costs or taxes.
- No dividend is paid.

### Formulas:
> $Call=S_0N(d_1)-Ke^{-rT}N(d_2)$

> $Put=Ke^{-rT}N(-d_2)-S_0N(-d_1)$

K: execution price<br>
r: risk-free interest rate<br>
T: time to expiration<br>
$\sigma$: volatility<br>
N: cumulative distribution function of the standard normal distribution<br>
$d_1=\frac{ln(\frac{S_0}{K})+(r+\frac{\sigma^2}{2})T}{\sigma\sqrt{T}}$<br>
$d_2=d_1-\sigma\sqrt{T}$

# Bi-Section 求 Volatility
選擇權定價以$S_0$, K, r, T, $\sigma$ 計算 Call / Put 價格，其中 $S_0$, K, r, T 都能從合約與市場取得，因此唯獨 $\sigma$ 需要估測。估測 $\sigma$ 的方法可以用歷史波動率或 Bi-section 方法計算 $\sigma$。<br>
透過使 $f(x)=BLS(S_0, K, r, T, \sigma)-Call$ 趨近於 0 的方式，找出最適合的 $\sigma$。<br>
<img src="../assets/bi-section.png" alt="bi-section" width="400"/>

# The Greek Letters
|||意義|Calls|Puts|
|-|-|-|-|-|
| Delta|$\frac{\partial Call}{\partial S_0}$ |指數S每動一單位，Call 的變動量|$N(d_1)$|$-N(-d_1)=N(d_1)-1$|
| Gamma|$\frac{\partial^2 Call}{\partial S_0^2}$ |指數S每動一單位，Delta 的變動量|$\frac{N'(d_1)}{S_0\sigma\sqrt{T-t}}$|$\frac{N'(d_1)}{S_0\sigma\sqrt{T-t}}$|
| Vega|$\frac{\partial Call}{\partial \sigma}$ |波動率每增加一單位，Call 價格的變動量|$S_0N'(d_1)\sqrt{T-t}$|$S_0N'(d_1)\sqrt{T-t}$|
| Theta|$\frac{\partial Call}{\partial T}$ |時間每減少一單位，Call 價格的變動量|$-\frac{S_0N'(d_1)\sigma}{2\sqrt{T-t}}-rKe^{-r(T-t)}N(d_2)$|$-\frac{S_0N'(-d_1)\sigma}{2\sqrt{T-t}}+rKe^{-r(T-t)}N(-d_2)$|
| Rho|$\frac{\partial Call}{\partial r}$ |無風險利率每增加一單位，Call 價格的變動量|$K(T-t)e^{-r(T-t)}N(d_2)$|$-K(T-t)e^{-r(T-t)}N(-d_2)$|

使用差分計算偏微分：<br>
||偏微分|差分|
|-|-|-|
| Delta|$\frac{\partial Call}{\partial S_0}$ |$\approx\frac{C(S_0+\Delta S)-C(S_0-\Delta S)}{2\Delta S}$|
| Gamma|$\frac{\partial^2 Call}{\partial S_0^2}$ | $\approx\frac{C(S_0+\Delta S)-2C(S_0)+C(S_0-\Delta S)}{\Delta S^2}$ |
| Vega|$\frac{\partial Call}{\partial \sigma}$ |$\approx\frac{C(\sigma+\Delta \sigma)-C(\sigma-\Delta \sigma)}{2\Delta \sigma}$|
| Theta|$\frac{\partial Call}{\partial T}$ |$\approx\frac{C(T-\Delta T)-C(T+\Delta T)}{2\Delta T}$|
| Rho|$\frac{\partial Call}{\partial r}$ |$\approx\frac{C(r+\Delta r)-C(r-\Delta r)}{2\Delta r}$|

In [6]:
import math
from scipy.stats import norm

In [7]:
def bls(S0, K, r, T, v):
    """
    Calculate the Black-Scholes option price for a European call and put option.
    
        Parameters:
        S0 : Current stock price
        K : Strike price
        r : Risk-free interest rate (annualized)
        T : Time to expiration (in years)
        v : Volatility of the stock (annualized)
    """
    
    d1 = (math.log(S0 / K) + (r + 0.5 * v ** 2) * T) / (v * math.sqrt(T))
    d2 = d1 - v * math.sqrt(T)
    
    call = S0 * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)
    put = K * math.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)
    
    return call, put

call_price, put_price = bls(50, 40, 0.08, 2, 0.2)
call_price

np.float64(16.383741845158895)

In [8]:
def bls_bisection(S0, K, r, T, p, tol=1e-6):
    """
    Calculate the implied volatility using the bisection method.
    
    parameters:
    S0 : Current stock price
    K : Strike price
    r : Risk-free interest rate (annualized)
    T : Time to expiration (in years)
    p : Market price of the option (call or put)
    """

    vol_left = 1e-6
    vol_right = 1
    
    while (vol_right - vol_left > tol):
        vol_mid = (vol_left + vol_right) / 2
        call, _ = bls(S0, K, r, T, vol_mid)
        
        if call < p:
            vol_left = vol_mid
        else:
            vol_right = vol_mid
            
    return (vol_left + vol_right) / 2

bls_bisection(50, 40, 0.08, 2, call_price)

0.20000013242864606

In [9]:
def greeks_analysis(S0, K, r, T, v):
    """
    Calculate the Greeks for a European call and put option.
    
    parameters:
    S0 : Current stock price
    K : Strike price
    r : Risk-free interest rate (annualized)
    T : Time to expiration (in years)
    v : Volatility of the stock (annualized)
    """
    
    d1 = (math.log(S0 / K) + (r + 0.5 * v ** 2) * T) / (v * math.sqrt(T))
    d2 = d1 - v * math.sqrt(T)
    
    delta_call = norm.cdf(d1)
    delta_put = norm.cdf(d1) - 1
    
    gamma = norm.pdf(d1) / (S0 * v * math.sqrt(T))
    
    theta_call = (-S0 * norm.pdf(d1) * v / (2 * math.sqrt(T)) - r * K * math.exp(-r * T) * norm.cdf(d2))
    theta_put = (-S0 * norm.pdf(d1) * v / (2 * math.sqrt(T)) + r * K * math.exp(-r * T) * norm.cdf(-d2))
    
    vega = S0 * norm.pdf(d1) * math.sqrt(T)
    
    rho_call = K * T * math.exp(-r * T) * norm.cdf(d2)
    rho_put = -K * T * math.exp(-r * T) * norm.cdf(-d2)
    
    return {
        'delta_call': delta_call,
        'delta_put': delta_put,
        'gamma': gamma,
        'theta_call': theta_call,
        'theta_put': theta_put,
        'vega': vega,
        'rho_call': rho_call,
        'rho_put': rho_put
    }
    
greeks_analysis(50, 40, 0.08, 2, 0.2)

{'delta_call': np.float64(0.9326781720838812),
 'delta_put': np.float64(-0.06732182791611885),
 'gamma': np.float64(0.00921278926433243),
 'theta_call': np.float64(-2.8806528039394346),
 'theta_put': np.float64(-0.15379267924755796),
 'vega': np.float64(9.212789264332432),
 'rho_call': np.float64(60.50033351807032),
 'rho_put': np.float64(-7.67116959922659)}

In [10]:
def greeks_analysis_diff(S0, K, r, T, v):
    """
    Calculate the Greeks for a European call and put option using finite difference method.
    
    parameters:
    S0 : Current stock price
    K : Strike price
    r : Risk-free interest rate (annualized)
    T : Time to expiration (in years)
    v : Volatility of the stock (annualized)
    """
    
    _diff = 1e-6
    # 
    delta_call = (bls(S0 + _diff, K, r, T, v)[0] - bls(S0 - _diff, K, r, T, v)[0]) / (2 * _diff)
    delta_put = (bls(S0 + _diff, K, r, T, v)[1] - bls(S0 - _diff, K, r, T, v)[1]) / (2 * _diff)
    
    gamma = (bls(S0 + _diff, K, r, T, v)[0] - 2 * bls(S0, K, r, T, v)[0] + bls(S0 - _diff, K, r, T, v)[0]) / (_diff ** 2)
    
    theta_call = (bls(S0, K, r, T - _diff, v)[0] - bls(S0, K, r, T + _diff, v)[0]) / (2 * _diff)
    theta_put = (bls(S0, K, r, T - _diff, v)[1] - bls(S0, K, r, T + _diff, v)[1]) / (2 * _diff)
    
    vega = (bls(S0, K, r, T, v + _diff)[0] - bls(S0, K, r, T, v - _diff)[0]) / (2 * _diff)
    
    rho_call = (bls(S0, K, r + _diff, T, v)[0] - bls(S0, K, r - _diff, T, v)[0]) / (2 * _diff)
    rho_put = (bls(S0, K, r + _diff, T, v)[1] - bls(S0, K, r - _diff, T, v)[1]) / (2 * _diff)
    
    return {
        'delta_call': delta_call,
        'delta_put': delta_put,
        'gamma': gamma,
        'theta_call': theta_call,
        'theta_put': theta_put,
        'vega': vega,
        'rho_call': rho_call,
        'rho_put': rho_put
    }
    
greeks_analysis_diff(50, 40, 0.08, 2, 0.2)

{'delta_call': np.float64(0.9326781675866869),
 'delta_put': np.float64(-0.06732182655788677),
 'gamma': np.float64(0.010658141036401503),
 'theta_call': np.float64(-2.8806528042935042),
 'theta_put': np.float64(-0.15379268036852523),
 'vega': np.float64(9.212789269241739),
 'rho_call': np.float64(60.5003335181209),
 'rho_put': np.float64(-7.671169599543504)}